In [6]:
import json
import re
def fix_json_string(json_str):
    try:
        if json_str.startswith("Here is the output in JSON format:"):
            json_str = json_str[len("Here is the output in JSON format:"):].strip()
        if json_str.startswith("Here is the output in the required JSON format:"):
            json_str = json_str[len("Here is the output in the required JSON format:"):].strip()
        json_str = re.sub(r',\s*}', '}', json_str)
        return json_str
    except Exception as e:
        print(f"Error in fix_json_string: {e}")
        raise


def extract_tags_from_file(file_path, tag_dict):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for user in data:
        try:
            response_str = user.get("response", "").replace("\n", "").replace("    ", "")
            fixed_response = fix_json_string(response_str)

            for tag_type in tag_dict.keys():
                if f'"{tag_type}":' in fixed_response:
                    content = fixed_response.split(f'"{tag_type}":')[1].split('},')[0]
                    keys = json.loads(content + '}').keys()
                    tag_dict[tag_type].update(keys)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")

In [ ]:

tag_types = ["Category", "Style", "Era", "Theme", "Emotion"]
tag_dict = {tag: set() for tag in tag_types}
video_file = "/root/autodl-tmp/crossaug_data/cell/cell_v11_summary.json"
toy_file = "/root/autodl-tmp/crossaug_data/elec/elec_v11_summary.json"
extract_tags_from_file(video_file, tag_dict)
extract_tags_from_file(toy_file, tag_dict)

In [ ]:

output_base = "/root/autodl-tmp/crossaug_data/cell/tags/"
for tag_type, tag_set in tag_dict.items():
    tag_list = sorted(tag_set)
    tag_to_id = {tag: idx for idx, tag in enumerate(tag_list)}
    output_file = f"{output_base}cell_{tag_type.lower()}_tag_map.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(tag_to_id, f, ensure_ascii=False, indent=4)
    print(f"{tag_type} 标签映射已保存到 {output_file}")